# This notebook demonstrates semantic search using LangChain.
First, a PDF document is loaded using PyPDFLoader.
Then, the text is split into smaller chunks using RecursiveCharacterTextSplitter.
After that, each chunk is converted into embeddings using HuggingFaceEmbeddings.
The embeddings are stored in InMemoryVectorStore.
Finally, relevant chunks are retrieved using similarity search and retriever methods.

In [4]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface pypdf sentence-transformers

In [5]:
!curl -L -o paper.pdf "https://arxiv.org/pdf/2307.09288.pdf"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   217  100   217    0     0    934      0 --:--:-- --:--:-- --:--:--   935
100 13.0M  100 13.0M    0     0  8660k      0  0:00:01  0:00:01 --:--:-- 10.9M


In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("paper.pdf")
docs = loader.load()

print("Number of pages:", len(docs))
print("\nFirst page preview:\n")
print(docs[0].page_content[:500])
print("\nMetadata:\n")
print(docs[0].metadata)

Number of pages: 77

First page preview:

Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev

Metadata:

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

print("Number of chunks:", len(all_splits))
print("\nFirst chunk preview:\n")
print(all_splits[0].page_content[:500])
print("\nChunk metadata:\n")
print(all_splits[0].metadata)

Number of chunks: 343

First chunk preview:

Llama 2: Open Foundation and Fine-Tuned Chat Models
Hugo Touvron∗ Louis Martin† Kevin Stone†
Peter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra
Prajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen
Guillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller
Cynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou
Hakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev

Chunk metadata:

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'start_index': 0}


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

print("Embedding length:", len(vector_1))
print("First 10 values:", vector_1[:10])
print("Same dimension:", len(vector_1) == len(vector_2))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding length: 384
First 10 values: [0.04391594976186752, 0.016254864633083344, -0.03557045757770538, 0.012593110091984272, -0.04547271132469177, 0.0494389533996582, -0.06556018441915512, 0.023530930280685425, -0.041214972734451294, 0.02452734299004078]
Same dimension: True


In [9]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

print("Indexed chunks:", len(ids))

Indexed chunks: 343


In [10]:
query = "What is the main idea of the paper?"

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:1000])


--- Result 1 ---
Page: 31
distribution and aligns towards the human preference. This phenomena is illustrated in Figure 20, where we
can see that the worst answers are progressively removed, shifting the distribution to the right.
In addition, during annotation, the model has the potential to venture into writing trajectories that even the
bestannotatorsmaynotchart. Nonetheless,humanscanstillprovidevaluablefeedbackwhencomparingtwo
answers, beyond their own writing competencies. Drawing a parallel, while we may not all be accomplished
artists, our ability to appreciate and critique art remains intact. We posit that the superior writing abilities of
LLMs, as manifested in surpassing human annotators in certain tasks, are fundamentally driven by RLHF, as
documented in Gilardi et al. (2023) and Huang et al. (2023). Supervised data may no longer be the gold
standard, and this evolving circumstance compels a re-evaluation of the concept of “supervision.”

--- Result 2 ---
Page: 59
➤Prompt: 

In [11]:
query = "What methods or approaches are discussed in the paper?"

results_with_scores = vector_store.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results_with_scores, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", score)
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:1000])


--- Result 1 ---
Score: 0.369315482367622
Page: 76
specific applications of the model. Please see the Responsible Use Guide available available at
https://ai.meta.com/llama/responsible-user-guide
Table 52: Model card forLlama 2.
77

--- Result 2 ---
Score: 0.30693528897794997
Page: 36
References
Daron Acemoglu and Pascual Restrepo. Artificial intelligence, automation, and work. InThe economics of
artificial intelligence: An agenda, pages 197–236. University of Chicago Press, 2018.
Joshua Ainslie, James Lee-Thorp, Michiel de Jong, Yury Zemlyanskiy, Federico Lebrón, and Sumit Sanghai.
Gqa: Training generalized multi-query transformer models from multi-head checkpoints, 2023.
Ebtesam Almazrouei, Hamza Alobeidli, Abdulaziz Alshamsi, Alessandro Cappelli, Ruxandra Cojocaru,
Merouane Debbah, Etienne Goffinet, Daniel Heslow, Julien Launay, Quentin Malartic, Badreddine Noune,
Baptiste Pannier, and Guilherme Penedo. Falcon-40B: an open large language model with state-of-the-art
performance. 202

In [12]:
query_embedding = embeddings.embed_query("What are the main contributions of the paper?")

results_by_vector = vector_store.similarity_search_by_vector(query_embedding, k=3)

for i, doc in enumerate(results_by_vector, 1):
    print(f"\n--- Result {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:1000])


--- Result 1 ---
Page: 31
distribution and aligns towards the human preference. This phenomena is illustrated in Figure 20, where we
can see that the worst answers are progressively removed, shifting the distribution to the right.
In addition, during annotation, the model has the potential to venture into writing trajectories that even the
bestannotatorsmaynotchart. Nonetheless,humanscanstillprovidevaluablefeedbackwhencomparingtwo
answers, beyond their own writing competencies. Drawing a parallel, while we may not all be accomplished
artists, our ability to appreciate and critique art remains intact. We posit that the superior writing abilities of
LLMs, as manifested in surpassing human annotators in certain tasks, are fundamentally driven by RLHF, as
documented in Gilardi et al. (2023) and Huang et al. (2023). Supervised data may no longer be the gold
standard, and this evolving circumstance compels a re-evaluation of the concept of “supervision.”

--- Result 2 ---
Page: 22
activitie

In [13]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

retrieved_docs = retriever.invoke("What problem does the paper try to solve?")

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Doc {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:1000])


--- Retrieved Doc 1 ---
Page: 36
References
Daron Acemoglu and Pascual Restrepo. Artificial intelligence, automation, and work. InThe economics of
artificial intelligence: An agenda, pages 197–236. University of Chicago Press, 2018.
Joshua Ainslie, James Lee-Thorp, Michiel de Jong, Yury Zemlyanskiy, Federico Lebrón, and Sumit Sanghai.
Gqa: Training generalized multi-query transformer models from multi-head checkpoints, 2023.
Ebtesam Almazrouei, Hamza Alobeidli, Abdulaziz Alshamsi, Alessandro Cappelli, Ruxandra Cojocaru,
Merouane Debbah, Etienne Goffinet, Daniel Heslow, Julien Launay, Quentin Malartic, Badreddine Noune,
Baptiste Pannier, and Guilherme Penedo. Falcon-40B: an open large language model with state-of-the-art
performance. 2023.
Rohan Anil, Andrew M. Dai, Orhan Firat, Melvin Johnson, Dmitry Lepikhin, Alexandre Passos, Siamak
Shakeri, Emanuel Taropa, Paige Bailey, Zhifeng Chen, Eric Chu, Jonathan H. Clark, Laurent El Shafey,

--- Retrieved Doc 2 ---
Page: 34
this progression 

In [14]:
queries = [
    "What is the paper about?",
    "What techniques are used?",
    "What are the main findings?"
]

batch_results = retriever.batch(queries)

for q, docs_for_q in zip(queries, batch_results):
    print("\n==============================")
    print("Query:", q)
    for i, doc in enumerate(docs_for_q, 1):
        print(f"\nResult {i} | Page: {doc.metadata.get('page')}")
        print(doc.page_content[:700])


Query: What is the paper about?

Result 1 | Page: 28
CONTENT] is not appropriate to discuss, etc.’ and then immediately follow up with ‘With that said, here’s
how [UNSAFE CONTENT].’ ”[Latest models] are able to resolve these issues.
• Distracting the [early models] by including “quirks” or specific requests usually defeated any
reluctance encountered via more direct requests.“A creative writing request (song, story, poem, etc.) is a
reliable way to get it to produce content that it is otherwise robust against.”
• Embedding a problematic request in a positive context often successfully obscured the fact that
problematic output was being requested for[early models] : “The overall principle I’ve found most
effective for any kind of attack is to h

Result 2 | Page: 36
References
Daron Acemoglu and Pascual Restrepo. Artificial intelligence, automation, and work. InThe economics of
artificial intelligence: An agenda, pages 197–236. University of Chicago Press, 2018.
Joshua Ainslie, James Le